# SCAFFOLD DESIGN

## Download new target

In [ ]:
# imports
from fragalysis.widgets import download
from pathlib import Path
import shutil
from os import environ
import json
import mrich
mrich.patch_rich_jupyter_margins()

In [ ]:
# this is the destination you should download to:
bulk_targets_dir = Path(environ["BULK"]) / "TARGETS"
print(bulk_targets_dir)

In [ ]:
# start the download widget
download()

## Initialise Directories for New Cycle

In [ ]:
target_name = "Target name"  # Change this to match the target name in Fragalysis.
cycle_number = 1  # Increment this for each new design cycle for this target.

In [ ]:
# all XChem-FFF work is kept under $HOME2/XChem-FFF; each target gets its own
# subdirectory here, and each design cycle its own subdirectory within that
xchem_fff_dir = Path(environ["HOME2"]) / "XChem-FFF"
target_dir = xchem_fff_dir / target_name.lower()
cycle_dir = target_dir / f"cycle_{cycle_number:02}"
fragmenstein_dir = cycle_dir / "fragmenstein"
knitwork_dir = cycle_dir / "knitwork"
knitwork_pure_output_dir = knitwork_dir / "knitwork_pure_output"
knitwork_impure_output_dir = knitwork_dir / "knitwork_impure_output"
gnina_dir = cycle_dir / "gnina"
gnina_inputs_dir = gnina_dir / "inputs"
gnina_inputs_mocassin_dir =  gnina_inputs_dir / "mocassin"
gnina_inputs_fragmenstein_dir =  gnina_inputs_dir / "fragmenstein"
gnina_inputs_knitwork_pure_dir =  gnina_inputs_dir / "knitwork_pure"
gnina_inputs_knitwork_impure_dir =  gnina_inputs_dir / "knitwork_impure"
gnina_outputs_dir = gnina_dir / "outputs"
gnina_outputs_mocassin_dir =  gnina_outputs_dir / "mocassin"
gnina_outputs_fragmenstein_dir =  gnina_outputs_dir / "fragmenstein"
gnina_outputs_knitwork_pure_dir =  gnina_outputs_dir / "knitwork_pure"
gnina_outputs_knitwork_impure_dir =  gnina_outputs_dir / "knitwork_impure"
mocassin_dir = cycle_dir / "mocassin"
mocassin_inputs_dir = mocassin_dir / "mocassin_inputs"
mocassin_outputs_dir = mocassin_dir / "mocassin_outputs"

In [ ]:
# create directories
cycle_dir.mkdir(parents=True, exist_ok=True)
fragmenstein_dir.mkdir(parents=True, exist_ok=True)
gnina_dir.mkdir(parents=True, exist_ok=True)
gnina_inputs_dir.mkdir(parents=True, exist_ok=True)
gnina_inputs_mocassin_dir.mkdir(parents=True, exist_ok=True)
gnina_inputs_fragmenstein_dir.mkdir(parents=True, exist_ok=True)
gnina_inputs_knitwork_pure_dir.mkdir(parents=True, exist_ok=True)
gnina_inputs_knitwork_impure_dir.mkdir(parents=True, exist_ok=True)
gnina_outputs_dir.mkdir(parents=True, exist_ok=True)
gnina_outputs_mocassin_dir.mkdir(parents=True, exist_ok=True)
gnina_outputs_fragmenstein_dir.mkdir(parents=True, exist_ok=True)
gnina_outputs_knitwork_pure_dir.mkdir(parents=True, exist_ok=True) 
gnina_outputs_knitwork_impure_dir.mkdir(parents=True, exist_ok=True) 
knitwork_dir.mkdir(parents=True, exist_ok=True)
knitwork_pure_output_dir.mkdir(parents=True, exist_ok=True)
knitwork_impure_output_dir.mkdir(parents=True, exist_ok=True)
mocassin_dir.mkdir(parents=True, exist_ok=True)
mocassin_inputs_dir.mkdir(parents=True, exist_ok=True)
mocassin_outputs_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# copy aligned files from $BULK to new directory
shutil.copytree(bulk_targets_dir / target_name / "aligned_files", target_dir / "aligned_files")

Change the permission of your XChem-FFF directory to you and group only by running the following command:

In [ ]:
environ["TARGET_DIR"] = str(target_dir)

# Change 770 to 700 if you want to restrict access to the target directory to only the owner.
!chmod -R 770 $TARGET_DIR

## Run Bulkdock / HIPPO

In [ ]:
%load_ext autoreload
%autoreload 2
import hippo
from mrich import print
from pathlib import Path
from os import environ
import shutil
import molparse as mp
import plotly.express as px

In [ ]:
bulk_target_dir = Path(environ["BULK"]) / "TARGETS" / target_name
environ["TARGET_NAME"] = str(target_name)

In [ ]:
!python -m bulkdock setup $TARGET_NAME

This creates an SQLite in $BULK/TARGETS/<TARGET_NAME>


In [ ]:
animal = hippo.HIPPO(target_name, bulk_target_dir / f"{target_name}.sqlite")

## Create Fragmenstein and Knitwork Inputs

In [ ]:
# list available tags
animal.tags

In [ ]:
merge_tag = "<your_tag>" # change `<your_tag>` to the selected tag from the list above, as tagged in Fragalysis
fragment_hits = animal.poses(tag=merge_tag)
fragment_hits.write_sdf(cycle_dir / "hits.sdf")

In [ ]:
# check how many tagged compounds will be processed
fragment_hits

In [ ]:
# download reference apo PDB for Fragmenstein - need a fragmenstein directory
ref_pose = fragment_hits[0]
shutil.copy(ref_pose.delig_path, cycle_dir / "fragmenstein")

## Run Fragmenstein

In [ ]:
environ["FRAGMENSTEIN_DIR"] = str(fragmenstein_dir.resolve())
Path(environ["FRAGMENSTEIN_DIR"])

Make sure the cell bellow outputs the full path of the `fragmenstein` directory

In [ ]:
!echo $FRAGMENSTEIN_DIR

In [ ]:
!sbatch --job-name "fragmenstein" --chdir $FRAGMENSTEIN_DIR --mem 16000 $HOME2/slurm/run_bash.sh $XCHEM_FFF/scripts/run_fragmenstein.sh

Verify if your job is running with the `squeue` command:

In [ ]:
!squeue -u $USER

## Run Knitwork on Squonk


Knitwork has to be run on [Squonk](https://data-manager-ui.xchem.diamond.ac.uk/data-manager-ui). Follow the instructions below and in the [Documentation](https://xchem-fff.readthedocs.io/en/latest/design.html#knitwork).

__in your local terminal__  
```bash
scp -J <yourfedid>@ssh.diamond.ac.uk <yourfedid>@cepheus-slurm.diamond.ac.uk:/<path_to_hits_file>/hits.sdf /<local_path>/
```

Change `<yourfedid>`, `<path_to_hits_file>`, and `<local_path>` to your FedID, the full path where `hits.sdf` can be found (you can use `pwd` in the directory where `hits.sdf` is) and the local path on your machine where you want the file to be copied to, respectively.

__in Squonk terminal__  

Follow the instructions in the Documentation to set up Knitworkm then run the following command:

```bash
python -m knitwork fragment hits.sdf
python -m knitwork pure-merge
python -m knitwork impure-merge
```

Output SDFs are written to the `knitwork_output/` directory in your Squonk Project Data file system. Download the `pure_merges.sdf` and `impure_merges.sdf` files following the instruction in the Documentation

__in your local terminal__

Copy both files to `knitwork_pure_output/<target_name>_pure_merges.sdf` and
`knitwork_impure_output/<target_name>_impure_merges.sdf` respectively using scp. 

```bash
scp -J <yourfedid>@ssh.diamond.ac.uk /<local_path>/pure_merges.sdf <yourfedid>@cepheus-slurm.diamond.ac.uk:/<path_to_knitwork_dir>/knitwork_pure_output/<target_name>_pure_merges.sdf
```

Repeat for the impure merges

## Generate poses

Now you can generate the poses for the scaffolds you have just designed. Copy the `Pose_Generation_Template.ipynb` into the `XChem-FFF` directory in your `$HOME2` and following the Pose Generation [Documentation](https://xchem-fff.readthedocs.io/en/latest/pose_generation.html) 